# 机器学习入门教程：临床疾病预测

本教程将带你从零开始，完成一个完整的机器学习项目流程。

## 📋 学习目标

1. **数据加载与探索** — 了解数据的基本结构和统计特征
2. **数据可视化** — 通过图表发现数据中的模式和关系
3. **数据预处理** — 处理缺失值、编码分类变量、特征缩放
4. **模型训练与比较** — 训练多种模型并比较性能
5. **模型评估** — 使用多种指标全面评估模型
6. **特征重要性分析** — 理解哪些特征对预测最重要

## 🎯 问题定义

**任务**：根据患者的年龄、BMI、血压、血糖、胆固醇等临床指标，预测患者是否患病。

**类型**：二分类问题（患病 / 健康）

---
## 1️⃣ 环境准备与数据加载

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print("所有库加载成功！✅")

In [ ]:
df = pd.read_csv('../data/clinical_data.csv')

print(f"数据集形状: {df.shape}")
print(f"样本数量: {df.shape[0]}, 特征数量: {df.shape[1] - 2}")
print()

---
## 2️⃣ 数据探索 (EDA)

在训练模型之前，我们必须先了解数据。这一步叫做**探索性数据分析 (Exploratory Data Analysis)**。

In [ ]:
print("=== 数据前5行 ===")
df.head()

In [ ]:
print("=== 数据基本信息 ===")
df.info()

In [ ]:
print("=== 数值特征统计描述 ===")
df.describe()

In [ ]:
print("=== 缺失值检查 ===")
missing = df.isnull().sum()
print(missing)
print(f"\n总缺失值: {df.isnull().sum().sum()}")

In [ ]:
print("=== 目标变量分布 ===")
print(df['diagnosis'].value_counts())
print(f"\n患病率: {(df['diagnosis'] == '患病').mean() * 100:.1f}%")

### 💡 关键发现

- 数据集包含 500 条患者记录
- 没有缺失值（实际项目中很少这么幸运！）
- 目标变量存在一定的类别不平衡（患病 > 健康）
- 数值特征的范围差异较大，后续需要特征缩放

---
## 3️⃣ 数据可视化

可视化是理解数据最直观的方式。

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

numerical_features = ['age', 'bmi', 'blood_pressure_systolic', 'blood_glucose_fasting', 'cholesterol_total']
colors = {'患病': '#e74c3c', '健康': '#2ecc71'}

for idx, feature in enumerate(numerical_features):
    ax = axes[idx // 3][idx % 3]
    for label, color in colors.items():
        subset = df[df['diagnosis'] == label][feature]
        ax.hist(subset, bins=25, alpha=0.6, label=label, color=color, edgecolor='white')
    ax.set_title(f'{feature} 分布', fontsize=13)
    ax.set_xlabel(feature)
    ax.set_ylabel('频数')
    ax.legend()

axes[1][2].axis('off')
plt.suptitle('临床指标分布（按是否患病分组）', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

rate_exercise = df.groupby('exercise_frequency')['diagnosis'].apply(
    lambda x: (x == '患病').mean() * 100
).reindex(['从不', '偶尔', '经常', '每天'])

rate_exercise.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'], edgecolor='white')
axes[0].set_title('不同运动频率的患病率', fontsize=13)
axes[0].set_ylabel('患病率 (%)')
axes[0].set_xlabel('运动频率')
axes[0].tick_params(axis='x', rotation=0)

rate_smoking = df.groupby('smoking_status')['diagnosis'].apply(
    lambda x: (x == '患病').mean() * 100
).reindex(['从不', '已戒烟', '仍在吸烟'])

rate_smoking.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white')
axes[1].set_title('不同吸烟状态的患病率', fontsize=13)
axes[1].set_ylabel('患病率 (%)')
axes[1].set_xlabel('吸烟状态')
axes[1].tick_params(axis='x', rotation=0)

rate_family = df.groupby('family_history')['diagnosis'].apply(
    lambda x: (x == '患病').mean() * 100
).reindex(['无', '有'])

rate_family.plot(kind='bar', ax=axes[2], color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[2].set_title('有无家族史的患病率', fontsize=13)
axes[2].set_ylabel('患病率 (%)')
axes[2].set_xlabel('家族史')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
numerical_df = df[numerical_features + ['diagnosis']].copy()
numerical_df['diagnosis_encoded'] = (numerical_df['diagnosis'] == '患病').astype(int)
corr_matrix = numerical_df.drop(columns=['diagnosis']).corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlBu_r', center=0,
            fmt='.3f', square=True, linewidths=0.5)
plt.title('临床指标相关性热力图', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, feature in enumerate(numerical_features):
    ax = axes[idx // 3][idx % 3]
    sns.boxplot(data=df, x='diagnosis', y=feature, ax=ax, palette=colors)
    ax.set_title(f'{feature} vs 患病/健康', fontsize=13)
    ax.set_xlabel('')

axes[1][2].axis('off')
plt.suptitle('临床指标与目标变量的箱线图', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4️⃣ 数据预处理

机器学习模型只能处理**数值型**输入。我们需要：
1. 编码分类变量
2. 划分训练集和测试集
3. 特征缩放

In [ ]:
df_model = df.drop(columns=['patient_id']).copy()

label_encoders = {}
categorical_cols = ['exercise_frequency', 'smoking_status', 'family_history']

for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le
    print(f"{col} 编码映射: {dict(zip(le.classes_, le.transform(le.classes_)))}")

df_model['diagnosis'] = (df_model['diagnosis'] == '患病').astype(int)

print(f"\n编码后数据形状: {df_model.shape}")
df_model.head()

In [ ]:
X = df_model.drop(columns=['diagnosis'])
y = df_model['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"训练集大小: {X_train.shape[0]}")
print(f"测试集大小: {X_test.shape[0]}")
print(f"训练集患病率: {y_train.mean() * 100:.1f}%")
print(f"测试集患病率: {y_test.mean() * 100:.1f}%")

In [ ]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

print("特征缩放完成！")
print(f"\n缩放后训练集统计:")
X_train_scaled[numerical_features].describe().round(2)

### 💡 预处理要点

- **LabelEncoder**：将分类变量转为数字（如 "有"→1, "无"→0）
- **train_test_split**：80% 训练，20% 测试，使用 stratify 保持类别比例
- **StandardScaler**：将数值特征缩放到均值=0、标准差=1，避免量纲影响
  - ⚠️ 只在训练集上 fit，然后 transform 测试集（防止数据泄露！）

---
## 5️⃣ 模型训练与比较

我们将训练 5 种不同的分类模型，比较它们的性能。

In [ ]:
models = {
    '逻辑回归': LogisticRegression(random_state=42, max_iter=1000),
    '决策树': DecisionTreeClassifier(random_state=42, max_depth=5),
    '随机森林': RandomForestClassifier(random_state=42, n_estimators=100),
    'SVM': SVC(random_state=42, probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }
    
    print(f"{name}: 准确率={results[name]['accuracy']:.4f}, "
          f"F1={results[name]['f1']:.4f}, "
          f"交叉验证={results[name]['cv_mean']:.4f}(±{results[name]['cv_std']:.4f})")

In [ ]:
results_df = pd.DataFrame({
    name: {
        '准确率': r['accuracy'],
        '精确率': r['precision'],
        '召回率': r['recall'],
        'F1分数': r['f1'],
        '交叉验证均值': r['cv_mean']
    }
    for name, r in results.items()
}).T

print("=== 模型性能比较 ===")
results_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrics = ['准确率', '精确率', '召回率', 'F1分数']
results_df[metrics].plot(kind='bar', ax=axes[0], edgecolor='white')
axes[0].set_title('模型性能指标对比', fontsize=14, fontweight='bold')
axes[0].set_ylabel('分数')
axes[0].set_ylim(0.5, 1.05)
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(loc='lower right')

x_pos = range(len(results))
cv_means = [r['cv_mean'] for r in results.values()]
cv_stds = [r['cv_std'] for r in results.values()]
model_names = list(results.keys())

axes[1].bar(x_pos, cv_means, yerr=cv_stds, capsize=5,
            color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'],
            edgecolor='white', alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(model_names)
axes[1].set_title('5折交叉验证准确率', fontsize=14, fontweight='bold')
axes[1].set_ylabel('准确率')
axes[1].set_ylim(0.5, 1.05)

plt.tight_layout()
plt.show()

---
## 6️⃣ 模型深入评估

让我们选择表现最好的模型进行更深入的分析。

In [ ]:
best_model_name = max(results, key=lambda x: results[x]['f1'])
best_result = results[best_model_name]

print(f"最佳模型（按F1分数）: {best_model_name}")
print(f"\n分类报告:")
print(classification_report(y_test, best_result['y_pred'], 
                          target_names=['健康', '患病']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, best_result['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['健康', '患病'], yticklabels=['健康', '患病'])
axes[0].set_title(f'{best_model_name} - 混淆矩阵', fontsize=13, fontweight='bold')
axes[0].set_ylabel('实际值')
axes[0].set_xlabel('预测值')

for name, r in results.items():
    if hasattr(r['model'], 'predict_proba'):
        y_prob = r['model'].predict_proba(X_test_scaled)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)
        axes[1].plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_title('ROC 曲线对比', fontsize=13, fontweight='bold')
axes[1].set_xlabel('假正率 (FPR)')
axes[1].set_ylabel('真正率 (TPR)')
axes[1].legend(loc='lower right')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

### 💡 评估指标解读

| 指标 | 含义 | 临床意义 |
|------|------|----------|
| **准确率** | 预测正确的比例 | 整体诊断正确率 |
| **精确率** | 预测为患病中真正患病的比例 | 阳性预测值，减少误诊 |
| **召回率** | 真正患病中被正确预测的比例 | 灵敏度，减少漏诊 |
| **F1分数** | 精确率和召回率的调和平均 | 综合平衡指标 |
| **AUC** | ROC曲线下面积，衡量分类能力 | 模型区分能力 |

- **混淆矩阵**：直观展示预测 vs 实际的对应关系
- **ROC曲线**：越靠近左上角，模型越好；AUC=1 为完美分类器
- ⚠️ 在临床场景中，**召回率（灵敏度）**尤为重要——漏诊的代价远高于误诊

---
## 7️⃣ 特征重要性分析

了解哪些特征对预测最重要，可以帮助我们：
1. 理解模型的决策逻辑
2. 进行特征选择，简化模型
3. 获得临床洞察

In [ ]:
rf_model = results['随机森林']['model']
feature_names = X.columns
importances = rf_model.feature_importances_

importance_df = pd.DataFrame({
    '特征': feature_names,
    '重要性': importances
}).sort_values('重要性', ascending=True)

plt.figure(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(importance_df)))
plt.barh(importance_df['特征'], importance_df['重要性'], color=colors, edgecolor='white')
plt.title('随机森林 - 特征重要性排序', fontsize=14, fontweight='bold')
plt.xlabel('重要性分数')
plt.tight_layout()
plt.show()

print("特征重要性排名:")
for i, row in importance_df.iloc[::-1].iterrows():
    print(f"  {row['特征']}: {row['重要性']:.4f}")

In [ ]:
if hasattr(results['逻辑回归']['model'], 'coef_'):
    lr_coefs = results['逻辑回归']['model'].coef_[0]
    coef_df = pd.DataFrame({
        '特征': feature_names,
        '系数': lr_coefs
    }).sort_values('系数')
    
    plt.figure(figsize=(10, 6))
    colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['系数']]
    plt.barh(coef_df['特征'], coef_df['系数'], color=colors, edgecolor='white')
    plt.axvline(x=0, color='black', linewidth=0.8)
    plt.title('逻辑回归 - 特征系数（正=促进患病，负=降低风险）', fontsize=13, fontweight='bold')
    plt.xlabel('系数值')
    plt.tight_layout()
    plt.show()

---
## 8️⃣ 总结与练习

### 📌 本教程回顾

我们完成了一个完整的机器学习项目流程：

1. ✅ **数据探索** — 了解数据结构和分布
2. ✅ **数据可视化** — 发现临床指标与目标变量的关系
3. ✅ **数据预处理** — 编码、划分、缩放
4. ✅ **模型训练** — 5种算法对比
5. ✅ **模型评估** — 多指标全面评估
6. ✅ **特征分析** — 理解模型决策依据

### 🏋️ 练习题

1. **调整超参数**：尝试修改决策树的 `max_depth`，观察对过拟合的影响
2. **特征选择**：只用最重要的3个特征重新训练，比较性能变化
3. **处理类别不平衡**：使用 `class_weight='balanced'` 或 SMOTE 过采样
4. **尝试新模型**：引入 XGBoost 或 LightGBM
5. **学习曲线**：绘制不同训练集大小下的模型性能曲线

In [ ]:
print("🎉 恭喜完成机器学习入门教程！")
print("\n尝试修改上面的代码，探索更多可能性吧！")